# Classifier-Free Guidance

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

Classifier-free guidance trains a single conditional diffusion model that also randomly drops the condition (making it unconditional). At sample time we extrapolate from the unconditional toward the conditional prediction, sharpening alignment with the prompt.


## Mathematical Formulation

$$\tilde \epsilon(x_t, c) = (1 + w)\,\epsilon_\theta(x_t, c) - w\,\epsilon_\theta(x_t, \varnothing)$$

The guidance scale $w$ trades diversity ($w = 0$) for fidelity to the condition ($w \gg 0$).


## Implementation


In [ ]:
import torch
import torch.nn as nn


In [ ]:
class ConditionalEpsilon(nn.Module):
    """Stub: a real model takes (x_t, t, c). Here we just demonstrate the math."""
    def __init__(self):
        super().__init__()
    def forward(self, x_t, t, c):
        # Simulate a conditional prediction with a deterministic offset
        return x_t + (0.0 if c is None else 0.5) * torch.ones_like(x_t)

def cfg_step(model, x_t, t, c, w=2.0, p_drop=0.1):
    eps_uncond = model(x_t, t, c=None)
    eps_cond   = model(x_t, t, c=c)
    return (1 + w) * eps_cond - w * eps_uncond


## Experiment


In [ ]:
model = ConditionalEpsilon()
x = torch.zeros(3, 4)
for w in [0.0, 1.0, 3.0, 7.5]:
    print(f'w={w:>4}: ', cfg_step(model, x, t=torch.tensor([0]), c='dog', w=w).mean().item())


## Discussion

- During training drop the condition uniformly at random (~10–20% of batches) so the same network can run both branches.
- High $w$ amplifies prompt fidelity but loses diversity and can over-saturate samples.
- Almost all modern text-to-image models (Stable Diffusion, Imagen, DALL·E 3) use CFG with $w \in [1, 12]$.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
